<a href="https://colab.research.google.com/github/eduardo-illueca/icg-project/blob/main/PRANN/PRANN_definitive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Libraries:


In [ ]:
# ========================
# Manejo de datos
# ========================
import pandas as pd
import numpy as np
from collections import Counter
import joblib
import warnings

# ========================
# Visualización
# ========================
import matplotlib.pyplot as plt

# ========================
# Procesamiento de señales
# ========================
from scipy import interpolate
from scipy.signal import find_peaks

# ========================
# Machine Learning - Scikit-learn
# ========================
import sklearn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# ========================
# Deep Learning - TensorFlow / Keras
# ========================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


##General Algorithm:

In [ ]:
class ICGSubtypeClassifier:
    """
    Impedance Cardiography Complex Subtype Classifier using Pattern Recognition
    Artificial Neural Networks (PRANN) with divide-and-conquer approach.

    Based on: Benouar et al. (2021) "Classification of impedance cardiography
    dZ/dt complex subtypes using pattern recognition artificial neural networks"
    """

    def __init__(self, target_freq=257, window_size_sec=1):
        self.target_freq = target_freq
        self.window_size_sec = window_size_sec
        self.window_size_samples = target_freq * window_size_sec

        # Neural networks (now using manual CV)
        self.autoencoder = None
        self.prann1 = None
        self.subprann1 = None
        self.subprann2 = None

        # Scalers
        self.scaler = MinMaxScaler()

        # Training history
        self.training_history = {}

    def build_autoencoder(self, input_dim, hidden_neurons=10):
        """
        Build autoencoder for synthetic data generation.
        Two feedforward networks with logarithmic-sigmoid activation.
        """
        # Encoder
        encoder_input = layers.Input(shape=(input_dim,))
        encoded = layers.Dense(hidden_neurons, activation='relu', name='encoder_hidden')(encoder_input)

        # Noise
        noisy_input = layers.GaussianNoise(0.1)(encoder_input)

        # Decoder
        decoded = layers.Dense(input_dim, activation='sigmoid', name='decoder_output')(encoded)

        # Autoencoder model
        autoencoder = keras.Model(encoder_input, decoded, name='autoencoder')

        # Compile with scaled conjugate gradient equivalent (Adam optimizer)
        autoencoder.compile(optimizer='adam', loss='mse', metrics=['mse'])

        return autoencoder

    def generate_synthetic_data(self, x1z_segments, labels, balance_classes=True):
        """
        Generate synthetic data using autoencoder to balance classes.
        Train separate autoencoder for each subtype.
        """
        unique_labels = np.unique(labels)
        synthetic_segments = []
        synthetic_labels = []

        if balance_classes:
            # Find the maximum class count for balancing
            max_count = max([np.sum(labels == label) for label in unique_labels])

        for label in unique_labels:
            class_indices = np.where(labels == label)[0]
            class_segments = x1z_segments[class_indices]

            if len(class_segments) == 0:
                continue

            # Build and train autoencoder for this class
            autoencoder = self.build_autoencoder(class_segments.shape[1])

            print(f"Label {label}")

            # Train autoencoder
            #autoencoder.fit(class_segments, class_segments,
            #              epochs=100, batch_size=16, verbose=0,
            #              validation_split=0.2)

            # Generate synthetic data
            if balance_classes:
                n_synthetic = max_count - len(class_segments)

                if n_synthetic > 0:
                    # Generate synthetic samples


                    for r in range(0, n_synthetic, n_synthetic % len(class_segments)):
                        synthetic_samples = autoencoder.predict(class_segments[:n_synthetic % len(class_segments)])

                        synthetic_segments.extend(synthetic_samples)
                        synthetic_labels.extend([label] * len(synthetic_samples))

                    print(len(synthetic_segments), len(synthetic_labels))

            # Add original samples
            print("**********")
            synthetic_segments.extend(class_segments)
            synthetic_labels.extend([label] * len(class_segments))

        return np.array(synthetic_segments), np.array(synthetic_labels)

    def build_prann(self, input_dim, n_classes, hidden_neurons=32, alpha=0.01, dropout_rate=0.3):
        """
        Build Pattern Recognition Artificial Neural Network (PRANN).
        Two-layer feedforward network with sigmoid and softmax layers.
        """
        model = keras.Sequential([
            layers.Input(shape=(input_dim,)),

            # First hidden layer
            layers.Dense(hidden_neurons, activation='relu', name='hidden_layer_1'),
            layers.LeakyReLU(alpha),
            layers.BatchNormalization(),
            layers.Dropout(dropout_rate),

            # Optional: Second hidden layer (if needed)
            layers.Dense(hidden_neurons // 2, activation='relu', name='hidden_layer_2'),
            layers.LeakyReLU(alpha),
            layers.BatchNormalization(),
            layers.Dropout(dropout_rate),

            # Output layer
            layers.Dense(n_classes, activation='softmax', name='output_layer')
        ])

        # Compile with scaled conjugate gradient equivalent
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        return model

    def create_divide_conquer_targets(self, labels):
        """
        Create targets for divide-and-conquer approach:
        PRANN1: Groups subtypes based on YOZ wave similarity
        - P1C1: ABEXYOZ subtypes 0, 1
        - P1C2: ABEXYOZ subtypes 2, 3, 4
        - P1C3: ABEXYOZ subtype u (undefined) - discarded
        """
        prann1_targets = np.zeros_like(labels)

        for i, label in enumerate(labels):
            if label in [0, 1]:
                prann1_targets[i] = 0  # P1C1
            elif label in [2, 3, 4]:
                prann1_targets[i] = 1  # P1C2
            else:  # undefined or other
                prann1_targets[i] = 2  # P1C3 (will be discarded)

        return prann1_targets

    def train_prann_system(self, x1z_segments, labels, test_size=0.3, validation_size=0.15, hidden_neurons=32, alpha=0.01, dropout_rate=0.3, synthetic = True):
        """
        Train the complete PRANN system using divide-and-conquer approach with manual CV.
        """
        print("Starting PRANN system training with manual cross-validation...")


        print("Generating synthetic data...")
        # For now, using original data (uncomment synthetic generation if needed)
        if synthetic:
          synthetic_segments, synthetic_labels = self.generate_synthetic_data(x1z_segments, labels, balance_classes=True)
        else:
          synthetic_segments, synthetic_labels = x1z_segments, labels
        # Split data: 70% training, 15% validation, 15% testing

        X_temp, X_test, y_temp, y_test = train_test_split(
            synthetic_segments, synthetic_labels, test_size=test_size,
            stratify=synthetic_labels, random_state=42)

        val_size_adjusted = validation_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=val_size_adjusted,
            stratify=y_temp, random_state=42)

        # Create PRANN1 targets (divide-and-conquer first level)
        y_train_p1 = self.create_divide_conquer_targets(y_train)
        y_val_p1 = self.create_divide_conquer_targets(y_val)
        y_test_p1 = self.create_divide_conquer_targets(y_test)

        # Remove undefined class (P1C3) for PRANN1 training
        valid_mask_train = y_train_p1 != 2
        valid_mask_val = y_val_p1 != 2
        valid_mask_test = y_test_p1 != 2

        X_train_p1 = X_train[valid_mask_train]
        y_train_p1 = y_train_p1[valid_mask_train]
        X_val_p1 = X_val[valid_mask_val]
        y_val_p1 = y_val_p1[valid_mask_val]

        # Train PRANN1 with manual cross-validation
        print("Training PRANN1 with manual cross-validation...")

        early_stopping = keras.callbacks.EarlyStopping(
                    monitor='val_accuracy', patience=20, restore_best_weights=True)

        self.prann1 = self.build_prann(X_train_p1.shape[1], 2, hidden_neurons=hidden_neurons ,dropout_rate=dropout_rate, alpha=alpha)

        history_prann1 = self.prann1.fit(
            X_train_p1, y_train_p1,
            validation_data=(X_val_p1, y_val_p1),
            epochs=1000, batch_size=32, verbose=0,
            callbacks=[early_stopping]
        )

        self.training_history['PRANN1'] = history_prann1
        # Predict PRANN1 outputs for SubPRANN training
        train_pred_p1 = self.prann1.predict(X_train_p1, verbose=0)
        train_pred_p1_class = np.argmax(train_pred_p1, axis=1)

        # Train SubPRANN1 (for P1C1: subtypes 0, 1)
        mask_p1c1_train = (y_train_p1 == 0)
        if np.sum(mask_p1c1_train) > 0:
            print("Training SubPRANN1...")
            X_train_sub1 = X_train_p1[mask_p1c1_train]
            y_train_sub1 = y_train[valid_mask_train][mask_p1c1_train]

            # Only include subtypes 0 and 1
            valid_sub1 = np.isin(y_train_sub1, [0, 1])
            X_train_sub1 = X_train_sub1[valid_sub1]
            y_train_sub1 = y_train_sub1[valid_sub1]

            if len(np.unique(y_train_sub1)) > 1:
                self.subprann1 = self.build_prann(X_train_sub1.shape[1], len(np.unique(y_train_sub1)), hidden_neurons=hidden_neurons ,dropout_rate=dropout_rate, alpha=alpha)

                # Create validation set for SubPRANN1
                X_train_sub1_split, X_val_sub1, y_train_sub1_split, y_val_sub1 = train_test_split(
                    X_train_sub1, y_train_sub1, test_size=0.2, stratify=y_train_sub1, random_state=42)

                early_stopping = keras.callbacks.EarlyStopping(
                    monitor='val_accuracy', patience=30, restore_best_weights=True)

                history_sub1 = self.subprann1.fit(
                    X_train_sub1_split, y_train_sub1_split,
                    validation_data=(X_val_sub1, y_val_sub1),
                    epochs=1000, batch_size=32, verbose=0,
                    callbacks=[early_stopping]
                )
                self.training_history['SubPRANN1'] = history_sub1

        # Train SubPRANN2 (for P1C2: subtypes 2, 3, 4)
        mask_p1c2_train = (y_train_p1 == 1)
        if np.sum(mask_p1c2_train) > 0:
            print("Training SubPRANN2...")
            X_train_sub2 = X_train_p1[mask_p1c2_train]
            y_train_sub2 = y_train[valid_mask_train][mask_p1c2_train]

            # Only include subtypes 2, 3, and 4
            valid_sub2 = np.isin(y_train_sub2, [2, 3, 4])
            X_train_sub2 = X_train_sub2[valid_sub2]
            y_train_sub2 = y_train_sub2[valid_sub2] - 2

            if len(np.unique(y_train_sub2)) > 1:
                self.subprann2 = self.build_prann(X_train_sub2.shape[1], len(np.unique(y_train_sub2)), hidden_neurons=hidden_neurons ,dropout_rate=dropout_rate, alpha=alpha)

                # Create validation set for SubPRANN2
                X_train_sub2_split, X_val_sub2, y_train_sub2_split, y_val_sub2 = train_test_split(
                    X_train_sub2, y_train_sub2, test_size=0.2, stratify=y_train_sub2, random_state=42)

                early_stopping = keras.callbacks.EarlyStopping(
                    monitor='val_accuracy', patience=30, restore_best_weights=True)

                history_sub2 = self.subprann2.fit(
                    X_train_sub2_split, y_train_sub2_split,
                    validation_data=(X_val_sub2, y_val_sub2),
                    epochs=1000, batch_size=32, verbose=0,
                    callbacks=[early_stopping]
                )
                self.training_history['SubPRANN2'] = history_sub2

        # Evaluate system on test set
        print("Evaluating PRANN system...")
        test_predictions = self.predict(X_test)
        test_accuracy = accuracy_score(y_test, test_predictions)

        print(f"Overall test accuracy: {test_accuracy:.3f}")

        return {
            'test_accuracy': test_accuracy,
            'test_data': (X_test, y_test),
            'predictions': test_predictions
        }

    def predict(self, X):
        """
        Make predictions using the complete PRANN system.
        """
        if self.prann1 is None:
            raise ValueError("PRANN system not trained. Call train_prann_system first.")

        # Step 1: PRANN1 classification
        prann1_pred = self.prann1.predict(X, verbose=0)
        prann1_classes = np.argmax(prann1_pred, axis=1)

        final_predictions = np.full(len(X), -1)  # Initialize with invalid class

        # Step 2: SubPRANN1 for P1C1 samples
        if self.subprann1 is not None:
            p1c1_mask = prann1_classes == 0
            if np.sum(p1c1_mask) > 0:
                sub1_pred = self.subprann1.predict(X[p1c1_mask], verbose=0)
                final_predictions[p1c1_mask] = np.argmax(sub1_pred, axis=1)

        # Step 3: SubPRANN2 for P1C2 samples
        if self.subprann2 is not None:
            p1c2_mask = prann1_classes == 1
            if np.sum(p1c2_mask) > 0:
                sub2_pred = self.subprann2.predict(X[p1c2_mask], verbose=0)
                sub2_classes = np.argmax(sub2_pred, axis=1)
                # Map back to original class labels (2, 3, 4)
                final_predictions[p1c2_mask] = sub2_classes + 2

        return final_predictions

    def evaluate_performance(self, y_true, y_pred):
        """
        Calculate comprehensive performance metrics.
        """
        accuracy = accuracy_score(y_true, y_pred)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0)

        # Per-class metrics
        precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
            y_true, y_pred, average=None, zero_division=0)

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'precision_per_class': precision_per_class,
            'recall_per_class': recall_per_class,
            'f1_per_class': f1_per_class,
            'confusion_matrix': confusion_matrix(y_true, y_pred)
        }

    def plot_training_history(self):
        """
        Plot training history for all networks.
        """
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))

        networks = ['PRANN1', 'SubPRANN1', 'SubPRANN2']

        for i, network in enumerate(networks):
            if network in self.training_history:
                history = self.training_history[network]

                # Accuracy plot
                axes[0, i].plot(history.history['accuracy'], label='Training')
                axes[0, i].plot(history.history['val_accuracy'], label='Validation')
                axes[0, i].set_title(f'{network} Accuracy')
                axes[0, i].set_xlabel('Epoch')
                axes[0, i].set_ylabel('Accuracy')
                axes[0, i].legend()
                axes[0, i].grid(True)

                # Loss plot
                axes[1, i].plot(history.history['loss'], label='Training')
                axes[1, i].plot(history.history['val_loss'], label='Validation')
                axes[1, i].set_title(f'{network} Loss')
                axes[1, i].set_xlabel('Epoch')
                axes[1, i].set_ylabel('Loss')
                axes[1, i].legend()
                axes[1, i].grid(True)
            else:
                axes[0, i].text(0.5, 0.5, f'{network}\nNot Trained',
                               ha='center', va='center', transform=axes[0, i].transAxes)
                axes[1, i].text(0.5, 0.5, f'{network}\nNot Trained',
                               ha='center', va='center', transform=axes[1, i].transAxes)

        plt.tight_layout()
        plt.show()


##Training HC K=5:

###Generic:

In [ ]:
classifier = ICGSubtypeClassifier()

data = pd.read_csv('heartCycle_full.csv').iloc[:, :]
aligned=pd.read_csv('HC_aligned_assignments_k=5.csv').iloc[:,:]
data['Cluster']=aligned['definitive_cluster']

train = data.iloc[:, :]

all_x1z_segments = np.array(train.iloc[:, 2:-1])
all_labels = np.array(train.iloc[:, -1])

print(f"Total processed segments: {len(all_x1z_segments)}")
print(f"Segment shape: {all_x1z_segments.shape}")
print(f"Label distribution: {np.bincount(all_labels)}")

# Parámetros reducidos para ejecución rápida
parameters = {
    "hidden_neurons": [512],
    "dropout_rate": [0.2],
    "alpha": [0.00001]
}

metrics = {
    "hidden_neurons": [],
    "dropout_rate": [],
    "alpha": [],
    "accuracy": []
}

print("\nStarting quick hyperparameter search...")

for n in parameters["hidden_neurons"]:
    for d in parameters["dropout_rate"]:
        for a in parameters["alpha"]:
            print(f"\nTrying configuration: hidden_neurons={n}, dropout_rate={d}, alpha={a}")
            cv = classifier.train_prann_system(
                all_x1z_segments, all_labels,
                test_size=0.1,
                validation_size=0.1,
                hidden_neurons=n,
                dropout_rate=d,
                alpha=a,
                synthetic=True
            )

            metrics["hidden_neurons"].append(n)
            metrics["dropout_rate"].append(d)
            metrics["alpha"].append(a)
            metrics["accuracy"].append(cv['test_accuracy'])

            print(f"Test accuracy for this config: {cv['test_accuracy']:.3f}")

# Guardar resultados
df_metrics = pd.DataFrame(metrics)
df_metrics.to_csv("quick_cross_validation_results.csv", index=False)

# Mostrar el mejor resultado encontrado
best_idx = df_metrics['accuracy'].idxmax()
best_params = df_metrics.loc[best_idx]
print("\nBest configuration found:")
print(best_params)

# Entrenar modelo final con los mejores parámetros
print("\nTraining final model with best parameters...")
result = classifier.train_prann_system(
    all_x1z_segments, all_labels,
    test_size=0.1,
    validation_size=0.1,
    hidden_neurons=int(best_params['hidden_neurons']),
    dropout_rate=float(best_params['dropout_rate']),
    alpha=float(best_params['alpha']),
    synthetic=True
)

performance = classifier.evaluate_performance(result['test_data'][1], result['predictions'])

print("\nFinal Results:")
print(f"Accuracy: {performance['accuracy']:.3f}")
print(f"Precision: {performance['precision']:.3f}")
print(f"Recall: {performance['recall']:.3f}")
print(f"F1-Score: {performance['f1_score']:.3f}")
print("\nConfusion Matrix:")
print(performance['confusion_matrix'])

# Si quieres ver los gráficos de entrenamiento, descomenta la siguiente línea:
# classifier.plot_training_history()


joblib.dump(classifier, "trained_icg_classifier.joblib")
print("Model saved as trained_icg_classifier.joblib")

# Guardar el modelo entrenado para uso futuro
model_filename = "trained_icg_classifier.joblib"
joblib.dump(classifier, model_filename)
print(f"\nModel saved to {model_filename}")

      Unnamed: 0   id         0         1         2         3         4  \
0              0    0  0.143926  0.059439 -0.011528 -0.070942 -0.121798   
1              1    1  0.224985  0.145768  0.077092  0.018294 -0.032098   
2              2    2  0.240889  0.146161  0.060711 -0.014502 -0.079758   
3              3    3  0.269712  0.187273  0.113702  0.047822 -0.012077   
4              4    4  0.210549  0.132366  0.064060  0.004799 -0.046800   
...          ...  ...       ...       ...       ...       ...       ...   
1565          95  995  0.842789  0.759481  0.666354  0.567843  0.467986   
1566          96  996 -0.125764 -0.187695 -0.241283 -0.285707 -0.320756   
1567          97  997 -0.102533 -0.169771 -0.227675 -0.275402 -0.312550   
1568          98  998 -0.056723 -0.114737 -0.168048 -0.217230 -0.262193   
1569          99  999 -0.023329 -0.083327 -0.138306 -0.187839 -0.231215   

             5         6         7  ...       108       109       110  \
0    -0.167225 -0.209714 -

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1285 1285
**********
Label 3
**********
Label 4


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
2382 2382
**********
Training PRANN1 with manual cross-validation...
Training SubPRANN1...
Training SubPRANN2...
Evaluating PRANN system...
Overall test accuracy: 0.710
Test accuracy for this config: 0.710

Best configuration found:
hidden_neurons    512.000000
dropout_rate        0.200000
alpha               0.000010
accuracy            0.709924
Name: 0, dtype: float64

Training final model with best parameters...
Starting PRANN system training with manual cross-validation...
Generating synthetic data...
Label 0
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
251 251
**********
Label 1
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
761 761
**********
Label 2
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1285 1285
**********
Label 3
**********
Label 4
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/ste

###Cluster 0:

In [ ]:
classifier = ICGSubtypeClassifier()

data = pd.read_csv('heartCycle_full.csv').iloc[:, :]
aligned=pd.read_csv('HC_aligned_assignments_k=5.csv').iloc[:,:]
data['Cluster']=aligned['definitive_cluster']
data=data[data['Cluster']==0]

train = data.iloc[:, :]

all_x1z_segments = np.array(train.iloc[:, 2:-2])
print(all_x1z_segments)
all_labels = np.array(train.iloc[:, -2])

print(f"Total processed segments: {len(all_x1z_segments)}")
print(f"Segment shape: {all_x1z_segments.shape}")
print(f"Label distribution: {np.bincount(all_labels)}")

# Mostrar distribución de subtipos para los segmentos del cluster 0
unique, counts = np.unique(all_labels, return_counts=True)
total = len(all_labels)

print("\nDistribución de subtipos para los segmentos del cluster 0:")
for label, count in zip(unique, counts):
    percentage = (count / total) * 100
    print(f"Subtipo {label}: {count} segmentos ({percentage:.2f}%)")

# Parámetros reducidos para ejecución rápida
parameters = {
    "hidden_neurons": [128],
    "dropout_rate": [0.2, 0.3],
    "alpha": [0.00001]
}

metrics = {
    "hidden_neurons": [],
    "dropout_rate": [],
    "alpha": [],
    "accuracy": []
}

print("\nStarting quick hyperparameter search...")

for n in parameters["hidden_neurons"]:
    for d in parameters["dropout_rate"]:
        for a in parameters["alpha"]:
            print(f"\nTrying configuration: hidden_neurons={n}, dropout_rate={d}, alpha={a}")
            cv = classifier.train_prann_system(
                all_x1z_segments, all_labels,
                test_size=0.1,
                validation_size=0.1,
                hidden_neurons=n,
                dropout_rate=d,
                alpha=a,
                synthetic=True
            )

            metrics["hidden_neurons"].append(n)
            metrics["dropout_rate"].append(d)
            metrics["alpha"].append(a)
            metrics["accuracy"].append(cv['test_accuracy'])

            print(f"Test accuracy for this config: {cv['test_accuracy']:.3f}")

# Guardar resultados
df_metrics = pd.DataFrame(metrics)
df_metrics.to_csv("quick_cross_validation_results.csv", index=False)

# Mostrar el mejor resultado encontrado
best_idx = df_metrics['accuracy'].idxmax()
best_params = df_metrics.loc[best_idx]
print("\nBest configuration found:")
print(best_params)

# Entrenar modelo final con los mejores parámetros
print("\nTraining final model with best parameters...")
result = classifier.train_prann_system(
    all_x1z_segments, all_labels,
    test_size=0.1,
    validation_size=0.1,
    hidden_neurons=int(best_params['hidden_neurons']),
    dropout_rate=float(best_params['dropout_rate']),
    alpha=float(best_params['alpha']),
    synthetic=True
)

performance = classifier.evaluate_performance(result['test_data'][1], result['predictions'])

print("\nFinal Results:")
print(f"Accuracy: {performance['accuracy']:.3f}")
print(f"Precision: {performance['precision']:.3f}")
print(f"Recall: {performance['recall']:.3f}")
print(f"F1-Score: {performance['f1_score']:.3f}")
print("\nConfusion Matrix:")
print(performance['confusion_matrix'])

# Si quieres ver los gráficos de entrenamiento, descomenta la siguiente línea:
# classifier.plot_training_history()


joblib.dump(classifier, "trained_icg_classifier_HC_0.joblib")
print("Model saved as trained_icg_classifier.joblib")

# Guardar el modelo entrenado para uso futuro
model_filename = "trained_icg_classifier_HC_0.joblib"
joblib.dump(classifier, model_filename)
print(f"\nModel saved to {model_filename}")

[[-0.09961334 -0.15177506 -0.19913226 ...  0.09325453  0.09516884
   0.09704341]
 [ 0.22836896  0.17194473  0.12164768 ... -0.25120048 -0.25737424
  -0.26262519]
 [ 0.07505979  0.02383698 -0.02222633 ... -0.17597759 -0.1771249
  -0.17946233]
 ...
 [ 0.65268041  0.5886161   0.52505614 ... -0.10216944 -0.09607213
  -0.0894667 ]
 [ 0.64362327  0.57456062  0.50309248 ... -0.57295305 -0.55697319
  -0.53904451]
 [ 0.61169763  0.52372068  0.43583101 ... -0.17095223 -0.13421305
  -0.09621737]]
Total processed segments: 268
Segment shape: (268, 116)
Label distribution: [ 30 113  13  51  61]

Distribución de subtipos para los segmentos del cluster 0:
Subtipo 0: 30 segmentos (11.19%)
Subtipo 1: 113 segmentos (42.16%)
Subtipo 2: 13 segmentos (4.85%)
Subtipo 3: 51 segmentos (19.03%)
Subtipo 4: 61 segmentos (22.76%)

Starting quick hyperparameter search...

Trying configuration: hidden_neurons=128, dropout_rate=0.2, alpha=1e-05
Starting PRANN system training with manual cross-validation...
Generatin

Training SubPRANN1...
Training SubPRANN2...
Evaluating PRANN system...


Overall test accuracy: 0.661
Test accuracy for this config: 0.661

Trying configuration: hidden_neurons=128, dropout_rate=0.3, alpha=1e-05
Starting PRANN system training with manual cross-validation...
Generating synthetic data...
Label 0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
92 92
**********
Label 1
**********
Label 2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
343 343
**********
Label 3
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━

###Cluster 1:

In [ ]:
classifier = ICGSubtypeClassifier()

data = pd.read_csv('heartCycle_full.csv').iloc[:, :]
aligned=pd.read_csv('HC_aligned_assignments_k=5.csv').iloc[:,:]
data['Cluster']=aligned['definitive_cluster']
data=data[data['Cluster']==1]

train = data.iloc[:, :]

all_x1z_segments = np.array(train.iloc[:, 2:-2])
print(all_x1z_segments)
all_labels = np.array(train.iloc[:, -2])

print(f"Total processed segments: {len(all_x1z_segments)}")
print(f"Segment shape: {all_x1z_segments.shape}")
print(f"Label distribution: {np.bincount(all_labels)}")

# Mostrar distribución de subtipos para los segmentos del cluster 1
unique, counts = np.unique(all_labels, return_counts=True)
total = len(all_labels)

print("\nDistribución de subtipos para los segmentos del cluster 1:")
for label, count in zip(unique, counts):
    percentage = (count / total) * 100
    print(f"Subtipo {label}: {count} segmentos ({percentage:.2f}%)")

# Parámetros reducidos para ejecución rápida
parameters = {
    "hidden_neurons": [256],
    "dropout_rate": [0.2],
    "alpha": [0.0001]
}

metrics = {
    "hidden_neurons": [],
    "dropout_rate": [],
    "alpha": [],
    "accuracy": []
}

print("\nStarting quick hyperparameter search...")

for n in parameters["hidden_neurons"]:
    for d in parameters["dropout_rate"]:
        for a in parameters["alpha"]:
            print(f"\nTrying configuration: hidden_neurons={n}, dropout_rate={d}, alpha={a}")
            cv = classifier.train_prann_system(
                all_x1z_segments, all_labels,
                test_size=0.1,
                validation_size=0.1,
                hidden_neurons=n,
                dropout_rate=d,
                alpha=a,
                synthetic=True
            )

            metrics["hidden_neurons"].append(n)
            metrics["dropout_rate"].append(d)
            metrics["alpha"].append(a)
            metrics["accuracy"].append(cv['test_accuracy'])

            print(f"Test accuracy for this config: {cv['test_accuracy']:.3f}")

# Guardar resultados
df_metrics = pd.DataFrame(metrics)
df_metrics.to_csv("quick_cross_validation_results.csv", index=False)

# Mostrar el mejor resultado encontrado
best_idx = df_metrics['accuracy'].idxmax()
best_params = df_metrics.loc[best_idx]
print("\nBest configuration found:")
print(best_params)

# Entrenar modelo final con los mejores parámetros
print("\nTraining final model with best parameters...")
result = classifier.train_prann_system(
    all_x1z_segments, all_labels,
    test_size=0.1,
    validation_size=0.1,
    hidden_neurons=int(best_params['hidden_neurons']),
    dropout_rate=float(best_params['dropout_rate']),
    alpha=float(best_params['alpha']),
    synthetic=True
)

performance = classifier.evaluate_performance(result['test_data'][1], result['predictions'])

print("\nFinal Results:")
print(f"Accuracy: {performance['accuracy']:.3f}")
print(f"Precision: {performance['precision']:.3f}")
print(f"Recall: {performance['recall']:.3f}")
print(f"F1-Score: {performance['f1_score']:.3f}")
print("\nConfusion Matrix:")
print(performance['confusion_matrix'])

# Si quieres ver los gráficos de entrenamiento, descomenta la siguiente línea:
# classifier.plot_training_history()


joblib.dump(classifier, "trained_icg_classifier_HC_1.joblib")
print("Model saved as trained_icg_classifier.joblib")

# Guardar el modelo entrenado para uso futuro
model_filename = "trained_icg_classifier_HC_1.joblib"
joblib.dump(classifier, model_filename)
print(f"\nModel saved to {model_filename}")

[[ 0.26710585  0.20192828  0.14588942 ... -0.0846511  -0.09437974
  -0.10458389]
 [ 0.01567147 -0.04338323 -0.09114584 ... -0.18988137 -0.18783659
  -0.18648527]
 [ 0.09223987  0.05151656  0.01336804 ... -0.22879203 -0.23372304
  -0.23822329]
 ...
 [ 0.73089939  0.62316425  0.50881594 ... -0.21705399 -0.21511135
  -0.21475427]
 [ 0.8422466   0.7450027   0.63380049 ... -0.1237666  -0.123307
  -0.11669841]
 [ 0.78390918  0.68636981  0.58173096 ... -0.18687794 -0.15908739
  -0.13218436]]
Total processed segments: 277
Segment shape: (277, 116)
Label distribution: [41 91 47 59 39]

Distribución de subtipos para los segmentos del cluster 1:
Subtipo 0: 41 segmentos (14.80%)
Subtipo 1: 91 segmentos (32.85%)
Subtipo 2: 47 segmentos (16.97%)
Subtipo 3: 59 segmentos (21.30%)
Subtipo 4: 39 segmentos (14.08%)

Starting quick hyperparameter search...

Trying configuration: hidden_neurons=256, dropout_rate=0.2, alpha=0.0001
Starting PRANN system training with manual cross-validation...
Generating syn

###Cluster 2:

In [ ]:
classifier = ICGSubtypeClassifier()

data = pd.read_csv('heartCycle_full.csv').iloc[:, :]
aligned=pd.read_csv('HC_aligned_assignments_k=5.csv').iloc[:,:]
data['Cluster']=aligned['definitive_cluster']
data=data[data['Cluster']==2]

train = data.iloc[:, :]

all_x1z_segments = np.array(train.iloc[:, 2:-2])
print(all_x1z_segments)
all_labels = np.array(train.iloc[:, -2])

print(f"Total processed segments: {len(all_x1z_segments)}")
print(f"Segment shape: {all_x1z_segments.shape}")
print(f"Label distribution: {np.bincount(all_labels)}")

# Mostrar distribución de subtipos para los segmentos del cluster 1
unique, counts = np.unique(all_labels, return_counts=True)
total = len(all_labels)

print("\nDistribución de subtipos para los segmentos del cluster 2:")
for label, count in zip(unique, counts):
    percentage = (count / total) * 100
    print(f"Subtipo {label}: {count} segmentos ({percentage:.2f}%)")

# Parámetros reducidos para ejecución rápida
parameters = {
    "hidden_neurons": [256],
    "dropout_rate": [0.2, 0.3],
    "alpha": [0.0001]
}

metrics = {
    "hidden_neurons": [],
    "dropout_rate": [],
    "alpha": [],
    "accuracy": []
}

print("\nStarting quick hyperparameter search...")

for n in parameters["hidden_neurons"]:
    for d in parameters["dropout_rate"]:
        for a in parameters["alpha"]:
            print(f"\nTrying configuration: hidden_neurons={n}, dropout_rate={d}, alpha={a}")
            cv = classifier.train_prann_system(
                all_x1z_segments, all_labels,
                test_size=0.1,
                validation_size=0.1,
                hidden_neurons=n,
                dropout_rate=d,
                alpha=a,
                synthetic=True
            )

            metrics["hidden_neurons"].append(n)
            metrics["dropout_rate"].append(d)
            metrics["alpha"].append(a)
            metrics["accuracy"].append(cv['test_accuracy'])

            print(f"Test accuracy for this config: {cv['test_accuracy']:.3f}")

# Guardar resultados
df_metrics = pd.DataFrame(metrics)
df_metrics.to_csv("quick_cross_validation_results.csv", index=False)

# Mostrar el mejor resultado encontrado
best_idx = df_metrics['accuracy'].idxmax()
best_params = df_metrics.loc[best_idx]
print("\nBest configuration found:")
print(best_params)

# Entrenar modelo final con los mejores parámetros
print("\nTraining final model with best parameters...")
result = classifier.train_prann_system(
    all_x1z_segments, all_labels,
    test_size=0.1,
    validation_size=0.1,
    hidden_neurons=int(best_params['hidden_neurons']),
    dropout_rate=float(best_params['dropout_rate']),
    alpha=float(best_params['alpha']),
    synthetic=True
)

performance = classifier.evaluate_performance(result['test_data'][1], result['predictions'])

print("\nFinal Results:")
print(f"Accuracy: {performance['accuracy']:.3f}")
print(f"Precision: {performance['precision']:.3f}")
print(f"Recall: {performance['recall']:.3f}")
print(f"F1-Score: {performance['f1_score']:.3f}")
print("\nConfusion Matrix:")
print(performance['confusion_matrix'])

# Si quieres ver los gráficos de entrenamiento, descomenta la siguiente línea:
# classifier.plot_training_history()


joblib.dump(classifier, "trained_icg_classifier_HC_2.joblib")
print("Model saved as trained_icg_classifier.joblib")

# Guardar el modelo entrenado para uso futuro
model_filename = "trained_icg_classifier_HC_2.joblib"
joblib.dump(classifier, model_filename)
print(f"\nModel saved to {model_filename}")

[[ 0.00243236 -0.06435506 -0.12378794 ... -0.20808332 -0.2067981
  -0.20251206]
 [-0.09572077 -0.15565186 -0.2076203  ... -0.2977151  -0.29276404
  -0.28488617]
 [-0.08924809 -0.14586898 -0.19609765 ... -0.21147911 -0.20848857
  -0.20322743]
 ...
 [ 0.81371672  0.71732101  0.61014222 ... -0.0372672  -0.02368217
  -0.01299814]
 [ 0.77416088  0.67226396  0.56117538 ... -0.05138331 -0.04697465
  -0.04064461]
 [ 0.75933377  0.66147545  0.55711458 ... -0.19051919 -0.17143525
  -0.14979649]]
Total processed segments: 272
Segment shape: (272, 116)
Label distribution: [67 93 19 32 61]

Distribución de subtipos para los segmentos del cluster 2:
Subtipo 0: 67 segmentos (24.63%)
Subtipo 1: 93 segmentos (34.19%)
Subtipo 2: 19 segmentos (6.99%)
Subtipo 3: 32 segmentos (11.76%)
Subtipo 4: 61 segmentos (22.43%)

Starting quick hyperparameter search...

Trying configuration: hidden_neurons=256, dropout_rate=0.2, alpha=0.0001
Starting PRANN system training with manual cross-validation...
Generating syn

###Cluster 3:

In [ ]:
classifier = ICGSubtypeClassifier()

data = pd.read_csv('heartCycle_full.csv').iloc[:, :]
aligned=pd.read_csv('HC_aligned_assignments_k=5.csv').iloc[:,:]
data['Cluster']=aligned['definitive_cluster']
data=data[data['Cluster']==3]

train = data.iloc[:, :]

all_x1z_segments = np.array(train.iloc[:, 2:-2])
print(all_x1z_segments)
all_labels = np.array(train.iloc[:, -2])

print(f"Total processed segments: {len(all_x1z_segments)}")
print(f"Segment shape: {all_x1z_segments.shape}")
print(f"Label distribution: {np.bincount(all_labels)}")

# Mostrar distribución de subtipos para los segmentos del cluster 1
unique, counts = np.unique(all_labels, return_counts=True)
total = len(all_labels)

print("\nDistribución de subtipos para los segmentos del cluster 3:")
for label, count in zip(unique, counts):
    percentage = (count / total) * 100
    print(f"Subtipo {label}: {count} segmentos ({percentage:.2f}%)")

# Parámetros reducidos para ejecución rápida
parameters = {
    "hidden_neurons": [128, 256],
    "dropout_rate": [0.3],
    "alpha": [0.00001]
}

metrics = {
    "hidden_neurons": [],
    "dropout_rate": [],
    "alpha": [],
    "accuracy": []
}

print("\nStarting quick hyperparameter search...")

for n in parameters["hidden_neurons"]:
    for d in parameters["dropout_rate"]:
        for a in parameters["alpha"]:
            print(f"\nTrying configuration: hidden_neurons={n}, dropout_rate={d}, alpha={a}")
            cv = classifier.train_prann_system(
                all_x1z_segments, all_labels,
                test_size=0.1,
                validation_size=0.1,
                hidden_neurons=n,
                dropout_rate=d,
                alpha=a,
                synthetic=True
            )

            metrics["hidden_neurons"].append(n)
            metrics["dropout_rate"].append(d)
            metrics["alpha"].append(a)
            metrics["accuracy"].append(cv['test_accuracy'])

            print(f"Test accuracy for this config: {cv['test_accuracy']:.3f}")

# Guardar resultados
df_metrics = pd.DataFrame(metrics)
df_metrics.to_csv("quick_cross_validation_results.csv", index=False)

# Mostrar el mejor resultado encontrado
best_idx = df_metrics['accuracy'].idxmax()
best_params = df_metrics.loc[best_idx]
print("\nBest configuration found:")
print(best_params)

# Entrenar modelo final con los mejores parámetros
print("\nTraining final model with best parameters...")
result = classifier.train_prann_system(
    all_x1z_segments, all_labels,
    test_size=0.1,
    validation_size=0.1,
    hidden_neurons=int(best_params['hidden_neurons']),
    dropout_rate=float(best_params['dropout_rate']),
    alpha=float(best_params['alpha']),
    synthetic=True
)

performance = classifier.evaluate_performance(result['test_data'][1], result['predictions'])

print("\nFinal Results:")
print(f"Accuracy: {performance['accuracy']:.3f}")
print(f"Precision: {performance['precision']:.3f}")
print(f"Recall: {performance['recall']:.3f}")
print(f"F1-Score: {performance['f1_score']:.3f}")
print("\nConfusion Matrix:")
print(performance['confusion_matrix'])

# Si quieres ver los gráficos de entrenamiento, descomenta la siguiente línea:
# classifier.plot_training_history()


joblib.dump(classifier, "trained_icg_classifier_HC_3.joblib")
print("Model saved as trained_icg_classifier.joblib")

# Guardar el modelo entrenado para uso futuro
model_filename = "trained_icg_classifier_HC_3.joblib"
joblib.dump(classifier, model_filename)
print(f"\nModel saved to {model_filename}")

[[ 0.14392625  0.0594386  -0.01152831 ... -0.04178719 -0.03317984
  -0.02491696]
 [ 0.22498499  0.14576761  0.07709239 ... -0.08197209 -0.08630354
  -0.09018165]
 [ 0.24088906  0.1461606   0.06071117 ... -0.01017592 -0.01618257
  -0.02133261]
 ...
 [-0.12576425 -0.18769531 -0.24128313 ... -0.17935278 -0.17629759
  -0.17047992]
 [-0.05672333 -0.11473739 -0.16804787 ... -0.24222801 -0.24477446
  -0.24834325]
 [-0.02332949 -0.08332667 -0.13830592 ... -0.15692833 -0.1374424
  -0.11829802]]
Total processed segments: 519
Segment shape: (519, 116)
Label distribution: [ 69 163  44 139 104]

Distribución de subtipos para los segmentos del cluster 3:
Subtipo 0: 69 segmentos (13.29%)
Subtipo 1: 163 segmentos (31.41%)
Subtipo 2: 44 segmentos (8.48%)
Subtipo 3: 139 segmentos (26.78%)
Subtipo 4: 104 segmentos (20.04%)

Starting quick hyperparameter search...

Trying configuration: hidden_neurons=128, dropout_rate=0.3, alpha=1e-05
Starting PRANN system training with manual cross-validation...
Generat

###Cluster 4:

In [ ]:
classifier = ICGSubtypeClassifier()

data = pd.read_csv('heartCycle_full.csv').iloc[:, :]
aligned=pd.read_csv('HC_aligned_assignments_k=5.csv').iloc[:,:]
data['Cluster']=aligned['definitive_cluster']
data=data[data['Cluster']==4]

train = data.iloc[:, :]

all_x1z_segments = np.array(train.iloc[:, 2:-2])
print(all_x1z_segments)
all_labels = np.array(train.iloc[:, -2])

print(f"Total processed segments: {len(all_x1z_segments)}")
print(f"Segment shape: {all_x1z_segments.shape}")
print(f"Label distribution: {np.bincount(all_labels)}")

# Mostrar distribución de subtipos para los segmentos del cluster 1
unique, counts = np.unique(all_labels, return_counts=True)
total = len(all_labels)

print("\nDistribución de subtipos para los segmentos del cluster 4:")
for label, count in zip(unique, counts):
    percentage = (count / total) * 100
    print(f"Subtipo {label}: {count} segmentos ({percentage:.2f}%)")

# Parámetros reducidos para ejecución rápida
parameters = {
    "hidden_neurons": [256],
    "dropout_rate": [0.2, 0.3],
    "alpha": [0.0001]
}

metrics = {
    "hidden_neurons": [],
    "dropout_rate": [],
    "alpha": [],
    "accuracy": []
}

print("\nStarting quick hyperparameter search...")

for n in parameters["hidden_neurons"]:
    for d in parameters["dropout_rate"]:
        for a in parameters["alpha"]:
            print(f"\nTrying configuration: hidden_neurons={n}, dropout_rate={d}, alpha={a}")
            cv = classifier.train_prann_system(
                all_x1z_segments, all_labels,
                test_size=0.1,
                validation_size=0.1,
                hidden_neurons=n,
                dropout_rate=d,
                alpha=a,
                synthetic=True
            )

            metrics["hidden_neurons"].append(n)
            metrics["dropout_rate"].append(d)
            metrics["alpha"].append(a)
            metrics["accuracy"].append(cv['test_accuracy'])

            print(f"Test accuracy for this config: {cv['test_accuracy']:.3f}")

# Guardar resultados
df_metrics = pd.DataFrame(metrics)
df_metrics.to_csv("quick_cross_validation_results.csv", index=False)

# Mostrar el mejor resultado encontrado
best_idx = df_metrics['accuracy'].idxmax()
best_params = df_metrics.loc[best_idx]
print("\nBest configuration found:")
print(best_params)

# Entrenar modelo final con los mejores parámetros
print("\nTraining final model with best parameters...")
result = classifier.train_prann_system(
    all_x1z_segments, all_labels,
    test_size=0.1,
    validation_size=0.1,
    hidden_neurons=int(best_params['hidden_neurons']),
    dropout_rate=float(best_params['dropout_rate']),
    alpha=float(best_params['alpha']),
    synthetic=True
)

performance = classifier.evaluate_performance(result['test_data'][1], result['predictions'])

print("\nFinal Results:")
print(f"Accuracy: {performance['accuracy']:.3f}")
print(f"Precision: {performance['precision']:.3f}")
print(f"Recall: {performance['recall']:.3f}")
print(f"F1-Score: {performance['f1_score']:.3f}")
print("\nConfusion Matrix:")
print(performance['confusion_matrix'])

# Si quieres ver los gráficos de entrenamiento, descomenta la siguiente línea:
# classifier.plot_training_history()


joblib.dump(classifier, "trained_icg_classifier_HC_4.joblib")
print("Model saved as trained_icg_classifier.joblib")

# Guardar el modelo entrenado para uso futuro
model_filename = "trained_icg_classifier_HC_4.joblib"
joblib.dump(classifier, model_filename)
print(f"\nModel saved to {model_filename}")

[[-0.06083327 -0.10716231 -0.142704   ... -0.11796935 -0.10302462
  -0.08909016]
 [-0.10477494 -0.16122665 -0.20864682 ... -0.28478998 -0.26402539
  -0.24223657]
 [ 0.04985643  0.00319008 -0.0367992  ... -0.27248243 -0.27155952
  -0.2697844 ]
 ...
 [ 0.8183707   0.73003946  0.6331223  ... -0.06299338 -0.05720335
  -0.05629345]
 [ 0.82306952  0.73748453  0.64272366 ... -0.12434281 -0.12020266
  -0.11509649]
 [-0.10253265 -0.16977079 -0.22767531 ... -0.12163553 -0.12004181
  -0.12068078]]
Total processed segments: 234
Segment shape: (234, 116)
Label distribution: [65 74 13 28 54]

Distribución de subtipos para los segmentos del cluster 4:
Subtipo 0: 65 segmentos (27.78%)
Subtipo 1: 74 segmentos (31.62%)
Subtipo 2: 13 segmentos (5.56%)
Subtipo 3: 28 segmentos (11.97%)
Subtipo 4: 54 segmentos (23.08%)

Starting quick hyperparameter search...

Trying configuration: hidden_neurons=256, dropout_rate=0.2, alpha=0.0001
Starting PRANN system training with manual cross-validation...
Generating sy

##HC K=4

In [ ]:
# Carga tus datos HeartCycle (ajusta el path o variable si ya lo tienes cargado)
heartcycle_data = pd.read_csv("heartCycle_full.csv")
aligned=pd.read_csv('HC_aligned_assignments_k=4.csv').iloc[:,:]
heartcycle_data['Cluster']=aligned['definitive_cluster']

# Filtramos solo clusters 0 a 3
clusters_to_consider = [0, 1, 2, 3]
data_filtered = heartcycle_data[heartcycle_data['Cluster'].isin(clusters_to_consider)]

# Agrupar por cluster y label y contar número de segmentos
counts = data_filtered.groupby(['Cluster', 'label']).size().reset_index(name='Count')

# Calcular porcentaje dentro de cada cluster
total_per_cluster = counts.groupby('Cluster')['Count'].transform('sum')
counts['Percentage'] = (counts['Count'] / total_per_cluster) * 100

# Mostrar resultados ordenados por cluster y label
counts = counts.sort_values(['Cluster', 'label'])

print(counts)

# Opcional: guardar a csv para análisis posterior
counts.to_csv("heartcycle_cluster_label_distribution.csv", index=False)


    Cluster  label  Count  Percentage
0         0      0    101   25.765306
1         0      1    133   33.928571
2         0      2     26    6.632653
3         0      3     46   11.734694
4         0      4     86   21.938776
5         1      0     41   14.642857
6         1      1     91   32.500000
7         1      2     49   17.500000
8         1      3     60   21.428571
9         1      4     39   13.928571
10        2      0     96   15.946844
11        2      1    187   31.063123
12        2      2     46    7.641196
13        2      3    149   24.750831
14        2      4    124   20.598007
15        3      0     34   11.486486
16        3      1    123   41.554054
17        3      2     15    5.067568
18        3      3     54   18.243243
19        3      4     70   23.648649


##HC K=3

In [ ]:

# Carga tus datos HeartCycle (ajusta el path o variable si ya lo tienes cargado)
heartcycle_data = pd.read_csv("heartCycle_full.csv")
aligned=pd.read_csv('HC_aligned_assignments_k=3.csv').iloc[:,:]
heartcycle_data['Cluster']=aligned['definitive_cluster']

# Filtramos solo clusters 0 a 2
clusters_to_consider = [0, 1, 2]
data_filtered = heartcycle_data[heartcycle_data['Cluster'].isin(clusters_to_consider)]

# Agrupar por cluster y label y contar número de segmentos
counts = data_filtered.groupby(['Cluster', 'label']).size().reset_index(name='Count')

# Calcular porcentaje dentro de cada cluster
total_per_cluster = counts.groupby('Cluster')['Count'].transform('sum')
counts['Percentage'] = (counts['Count'] / total_per_cluster) * 100

# Mostrar resultados ordenados por cluster y label
counts = counts.sort_values(['Cluster', 'label'])

print(counts)

# Opcional: guardar a csv para análisis posterior
counts.to_csv("heartcycle_cluster_label_distribution.csv", index=False)


    Cluster  label  Count  Percentage
0         0      0     66   12.222222
1         0      1    157   29.074074
2         0      2     71   13.148148
3         0      3    152   28.148148
4         0      4     94   17.407407
5         1      0    113   24.618736
6         1      1    156   33.986928
7         1      2     25    5.446623
8         1      3     58   12.636166
9         1      4    107   23.311547
10        2      0     93   16.287215
11        2      1    221   38.704028
12        2      2     40    7.005254
13        2      3     99   17.338004
14        2      4    118   20.665499


##ReBeat k=5

In [ ]:
# Cargar datos de ReBeat
rebeat_data = pd.read_csv('RB_aligned_assignments_k=5.csv')

# Procesar la columna 'Segment': convertir strings multilinea a listas de floats
def parse_segment(segment_str):
    segment_str = segment_str.strip().replace('[', '').replace(']', '')
    return np.array([float(x) for x in segment_str.split()])

def truncate_segment_keep_final(segment, keep_ratio=0.625):
    length = int(len(segment) * keep_ratio)
    return segment[-length:]  # Conserva solo el tramo final

rebeat_data['Parsed_Segment'] = rebeat_data['Segment'].apply(parse_segment)
rebeat_data['Parsed_Segment'] = rebeat_data['Parsed_Segment'].apply(truncate_segment_keep_final)

# Agrupar segmentos por cluster
clusters = rebeat_data['definitive_cluster'].unique()

# Preparar DataFrame para guardar los resultados
results = []

# Aplicar cada modelo de HeartCycle a cada cluster de ReBeat
for cluster_id in clusters:
    print(f"\nProcesando cluster ReBeat {cluster_id}...")

    # Seleccionar los segmentos de este cluster
    cluster_segments = rebeat_data[rebeat_data['definitive_cluster'] == cluster_id]['Parsed_Segment'].tolist()
    X_cluster = np.stack(cluster_segments)  # Convertimos a matriz

    for hc_model_id in range(5):  # Modelos de HeartCycle del 0 al 4
        model_path = f"trained_icg_classifier_HC_{hc_model_id}.joblib"
        print(f"  Aplicando modelo {model_path}...")

        classifier = joblib.load(model_path)

        # Realizar predicciones sobre los segmentos del cluster
        predictions = classifier.predict(X_cluster)

        # Calcular distribución de etiquetas
        label_counts = dict(Counter(predictions))
        total = sum(label_counts.values())

        # Calcular entropía de las predicciones
        entropy = -sum((count / total) * np.log2(count / total) for count in label_counts.values())

        # Guardar los resultados
        results.append({
            "rebeat_cluster": cluster_id,
            "hc_model": hc_model_id,
            "predicted_subtypes": label_counts,
            "entropy": entropy,
            "total_segments": total
        })

# Convertir resultados a DataFrame plano para exportar
rows = []
for entry in results:
    flat_row = {
        "ReBeat Cluster": entry['rebeat_cluster'],
        "HC Model": entry['hc_model'],
        "Entropy": entry['entropy'],
        "Total Segments": entry['total_segments']
    }
    for subtype, count in entry['predicted_subtypes'].items():
        flat_row[f"Subtype_{subtype}"] = count
    rows.append(flat_row)

df_results = pd.DataFrame(rows).fillna(0).sort_values(by=["ReBeat Cluster", "Entropy"])

# Guardar en CSV
df_results.to_csv("rebeat_cluster_predictions_by_HC_model.csv", index=False)
print("\n Resultados guardados en 'rebeat_cluster_predictions_by_HC_model.csv'")



Procesando cluster ReBeat 1...
  Aplicando modelo trained_icg_classifier_HC_0.joblib...
  Aplicando modelo trained_icg_classifier_HC_1.joblib...
  Aplicando modelo trained_icg_classifier_HC_2.joblib...
  Aplicando modelo trained_icg_classifier_HC_3.joblib...
  Aplicando modelo trained_icg_classifier_HC_4.joblib...

Procesando cluster ReBeat 0...
  Aplicando modelo trained_icg_classifier_HC_0.joblib...
  Aplicando modelo trained_icg_classifier_HC_1.joblib...
  Aplicando modelo trained_icg_classifier_HC_2.joblib...
  Aplicando modelo trained_icg_classifier_HC_3.joblib...
  Aplicando modelo trained_icg_classifier_HC_4.joblib...

Procesando cluster ReBeat 3...
  Aplicando modelo trained_icg_classifier_HC_0.joblib...
  Aplicando modelo trained_icg_classifier_HC_1.joblib...
  Aplicando modelo trained_icg_classifier_HC_2.joblib...
  Aplicando modelo trained_icg_classifier_HC_3.joblib...
  Aplicando modelo trained_icg_classifier_HC_4.joblib...

Procesando cluster ReBeat 4...
  Aplicando model

##ReBeat k=4

In [ ]:
# Cargar datos de ReBeat
rebeat_data = pd.read_csv('RB_aligned_assignments_k=4.csv')

# Procesar la columna 'Segment': convertir strings multilinea a listas de floats
def parse_segment(segment_str):
    segment_str = segment_str.strip().replace('[', '').replace(']', '')
    return np.array([float(x) for x in segment_str.split()])

def truncate_segment_keep_final(segment, keep_ratio=0.625):
    length = int(len(segment) * keep_ratio)
    return segment[-length:]  # Conserva solo el tramo final

rebeat_data['Parsed_Segment'] = rebeat_data['Segment'].apply(parse_segment)
rebeat_data['Parsed_Segment'] = rebeat_data['Parsed_Segment'].apply(truncate_segment_keep_final)

# Agrupar segmentos por cluster
clusters = rebeat_data['definitive_cluster'].unique()

# Preparar DataFrame para guardar los resultados
results = []

# Aplicar cada modelo de HeartCycle a cada cluster de ReBeat
for cluster_id in clusters:
    print(f"\nProcesando cluster ReBeat {cluster_id}...")

    # Seleccionar los segmentos de este cluster
    cluster_segments = rebeat_data[rebeat_data['definitive_cluster'] == cluster_id]['Parsed_Segment'].tolist()
    X_cluster = np.stack(cluster_segments)  # Convertimos a matriz

    for hc_model_id in range(5):  # Modelos de HeartCycle del 0 al 4
        model_path = f"trained_icg_classifier_HC_{hc_model_id}.joblib"
        print(f"  Aplicando modelo {model_path}...")

        classifier = joblib.load(model_path)

        # Realizar predicciones sobre los segmentos del cluster
        predictions = classifier.predict(X_cluster)

        # Calcular distribución de etiquetas
        label_counts = dict(Counter(predictions))
        total = sum(label_counts.values())

        # Calcular entropía de las predicciones
        entropy = -sum((count / total) * np.log2(count / total) for count in label_counts.values())

        # Guardar los resultados
        results.append({
            "rebeat_cluster": cluster_id,
            "hc_model": hc_model_id,
            "predicted_subtypes": label_counts,
            "entropy": entropy,
            "total_segments": total
        })

# Convertir resultados a DataFrame plano para exportar
rows = []
for entry in results:
    flat_row = {
        "ReBeat Cluster": entry['rebeat_cluster'],
        "HC Model": entry['hc_model'],
        "Entropy": entry['entropy'],
        "Total Segments": entry['total_segments']
    }
    for subtype, count in entry['predicted_subtypes'].items():
        flat_row[f"Subtype_{subtype}"] = count
    rows.append(flat_row)

df_results = pd.DataFrame(rows).fillna(0).sort_values(by=["ReBeat Cluster", "Entropy"])

# Guardar en CSV
df_results.to_csv("rebeat_cluster_predictions_by_HC_model_k4.csv", index=False)
print("\n Resultados guardados en 'rebeat_cluster_predictions_by_HC_model.csv'")



Procesando cluster ReBeat 1.0...
  Aplicando modelo trained_icg_classifier_HC_0.joblib...
  Aplicando modelo trained_icg_classifier_HC_1.joblib...
  Aplicando modelo trained_icg_classifier_HC_2.joblib...
  Aplicando modelo trained_icg_classifier_HC_3.joblib...
  Aplicando modelo trained_icg_classifier_HC_4.joblib...

Procesando cluster ReBeat 0.0...
  Aplicando modelo trained_icg_classifier_HC_0.joblib...
  Aplicando modelo trained_icg_classifier_HC_1.joblib...
  Aplicando modelo trained_icg_classifier_HC_2.joblib...
  Aplicando modelo trained_icg_classifier_HC_3.joblib...
  Aplicando modelo trained_icg_classifier_HC_4.joblib...

Procesando cluster ReBeat 2.0...
  Aplicando modelo trained_icg_classifier_HC_0.joblib...
  Aplicando modelo trained_icg_classifier_HC_1.joblib...
  Aplicando modelo trained_icg_classifier_HC_2.joblib...
  Aplicando modelo trained_icg_classifier_HC_3.joblib...
  Aplicando modelo trained_icg_classifier_HC_4.joblib...

Procesando cluster ReBeat 3.0...
  Aplican

##ReBeat k=3

In [ ]:
# Cargar datos de ReBeat
rebeat_data = pd.read_csv('RB_aligned_assignments_k=3.csv')

# Procesar la columna 'Segment': convertir strings multilinea a listas de floats
def parse_segment(segment_str):
    segment_str = segment_str.strip().replace('[', '').replace(']', '')
    return np.array([float(x) for x in segment_str.split()])

def truncate_segment_keep_final(segment, keep_ratio=0.625):
    length = int(len(segment) * keep_ratio)
    return segment[-length:]  # Conserva solo el tramo final

rebeat_data['Parsed_Segment'] = rebeat_data['Segment'].apply(parse_segment)
rebeat_data['Parsed_Segment'] = rebeat_data['Parsed_Segment'].apply(truncate_segment_keep_final)

# Agrupar segmentos por cluster
clusters = rebeat_data['definitive_cluster'].unique()

# Preparar DataFrame para guardar los resultados
results = []

# Aplicar cada modelo de HeartCycle a cada cluster de ReBeat
for cluster_id in clusters:
    print(f"\nProcesando cluster ReBeat {cluster_id}...")

    # Seleccionar los segmentos de este cluster
    cluster_segments = rebeat_data[rebeat_data['definitive_cluster'] == cluster_id]['Parsed_Segment'].tolist()
    X_cluster = np.stack(cluster_segments)  # Convertimos a matriz

    for hc_model_id in range(5):  # Modelos de HeartCycle del 0 al 4
        model_path = f"trained_icg_classifier_HC_{hc_model_id}.joblib"
        print(f"  Aplicando modelo {model_path}...")

        classifier = joblib.load(model_path)

        # Realizar predicciones sobre los segmentos del cluster
        predictions = classifier.predict(X_cluster)

        # Calcular distribución de etiquetas
        label_counts = dict(Counter(predictions))
        total = sum(label_counts.values())

        # Calcular entropía de las predicciones
        entropy = -sum((count / total) * np.log2(count / total) for count in label_counts.values())

        # Guardar los resultados
        results.append({
            "rebeat_cluster": cluster_id,
            "hc_model": hc_model_id,
            "predicted_subtypes": label_counts,
            "entropy": entropy,
            "total_segments": total
        })

# Convertir resultados a DataFrame plano para exportar
rows = []
for entry in results:
    flat_row = {
        "ReBeat Cluster": entry['rebeat_cluster'],
        "HC Model": entry['hc_model'],
        "Entropy": entry['entropy'],
        "Total Segments": entry['total_segments']
    }
    for subtype, count in entry['predicted_subtypes'].items():
        flat_row[f"Subtype_{subtype}"] = count
    rows.append(flat_row)

df_results = pd.DataFrame(rows).fillna(0).sort_values(by=["ReBeat Cluster", "Entropy"])

# Guardar en CSV
df_results.to_csv("rebeat_cluster_predictions_by_HC_model_k3.csv", index=False)
print("\n Resultados guardados en 'rebeat_cluster_predictions_by_HC_model.csv'")



Procesando cluster ReBeat 1...
  Aplicando modelo trained_icg_classifier_HC_0.joblib...
  Aplicando modelo trained_icg_classifier_HC_1.joblib...
  Aplicando modelo trained_icg_classifier_HC_2.joblib...
  Aplicando modelo trained_icg_classifier_HC_3.joblib...
  Aplicando modelo trained_icg_classifier_HC_4.joblib...

Procesando cluster ReBeat 0...
  Aplicando modelo trained_icg_classifier_HC_0.joblib...
  Aplicando modelo trained_icg_classifier_HC_1.joblib...
  Aplicando modelo trained_icg_classifier_HC_2.joblib...
  Aplicando modelo trained_icg_classifier_HC_3.joblib...
  Aplicando modelo trained_icg_classifier_HC_4.joblib...

Procesando cluster ReBeat 2...
  Aplicando modelo trained_icg_classifier_HC_0.joblib...
  Aplicando modelo trained_icg_classifier_HC_1.joblib...
  Aplicando modelo trained_icg_classifier_HC_2.joblib...
  Aplicando modelo trained_icg_classifier_HC_3.joblib...
  Aplicando modelo trained_icg_classifier_HC_4.joblib...

 Resultados guardados en 'rebeat_cluster_predict